## 🔰PyTorchでニューラルネットワーク基礎 #40 【GPT編・指示チューニング】

### 内容
* Qiitaの記事と連動しています
* フルスクラッチの超小型GPTタイプに指示チューニングを行ったものです
* 各種ファイルの保存先は環境によって適宜変更してください

### データについて　（詳細はQiita記事側）
* ライセンスの関係から改変データを公開できないので、データは適宜作成してもらえると幸いです。
* Livedoorニュースコーパスのlivedoor-hommeカテゴリを利用します。
* huggingfaceの　llm-book/livedoor-news-corpus　などから適宜ダウンロードしてください。
* SFT用のデータは「質問文」「応答文」のような形になります。livedoor-hommeカテゴリーの文章から1ファイルから1問だけ作成しました。
* 簡単な挨拶文も追加しています。

### トークナイザーについて
* tokenizer/livedoor_homme_tokenizer_8k.json
    * bytelevel BPEで構成した語彙数8kのtokenizer

### 事前学習済みのモデルについて
* model/homme_seq_512_bpe_8k.model
    * 第38回（sample_38.ipynb）で学習して保存したモデル
    * 系列長（seq_len=512）が512です。512トークンを超える場合はNG

### 指示チューニング済みのモデルについて
* homme_seq_512_bpe_8k_it.model
    * SFT用データを作成して学習したもの

### 注意点
* 汎用的な内容は生成不能
* 学習した内容、学習した文と類似文章が入力されるとうまく生成される
* 学習していない内容は、全くだめ
* モデルサイズ・データサイズが小さいので完全コピーに近い生成がみられるがこれが正常な状況
* 系列長を512とある程度長くしないと、指示チューニング時にトークン不足となる
* 事前学習だけを考える場合、GPUメモリとの兼ね合いで seq_len=128でも事前学習による生成可能


### 準備
* ライブラリーの読み込み
* シード固定の方法、こちらは結構適当なので参考にしないでください

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from pathlib import Path


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

import random
import numpy as np

def set_seed(seed=55):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)   # CPU + 全CUDAデバイスをまとめて設定する
    print(f"seedを設定: {seed}")
set_seed()

device=device(type='cuda')
seedを設定: 55


In [ ]:
data_filename = "./data/instruction_turning_data.jsonl"             # SFT用のデータここは作成してください
tokenizer_filename = "./tokenizer/livedoor_home_tokenizer_8k.json"
pretrain_filename = "model/seq_512_bpe_8k.model"
instruct_model_filename = "model/homme_seq_512_bpe_8k_it.model"

tokenizer = Tokenizer.from_file(tokenizer_filename)
print(f"size: {tokenizer.get_vocab_size()}")

size: 8000


## GPTタイプモデル

In [3]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 512   # 128トークンだとSFT時に少ない
        self.d_model = 256   # 512
        self.nhead = 8
        self.dim_feedforward = 4*self.d_model
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.eod_token_id = tokenizer.token_to_id("<eod>")
        
       
        # 学習データに関する設定
        self.context_size = self.seq_len         # 学習できる長さ
        self.context_stride = self.context_size  # 重なり具合の調整

        # 学習設定
        self.batch_size = 128
        self.learning_rate = 0.001  # これだとデフォルトと変わらない
        self.num_epochs = 100
        self.max_grad_norm = 1.0
        self.ignore_index = -100

    # 属性を追加・更新するメソッドを追加 設定時のタイポに注意だぞ〜
    def update(self, **kwargs):
        """渡されたキーワード引数で設定を動的に追加・更新する"""
        for key, value in kwargs.items():
            setattr(self, key, value)

In [4]:
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        
        # 埋め込み層 pad id 学習しない(今回ないはずだが)
        self.token_embedding = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer layers
        # TransformerEncoderだけど、右三角にmaskつけるのでマスク付き自己注意のタイプになる
        causal_transformer_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True     # 正規化の場所指定
        )
        self.transformer = nn.TransformerEncoder(causal_transformer_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # 最後の出力に向けた正規化とFC層　最終的に単語数になる
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.fc = nn.Linear(config.d_model, config.vocab_size,bias=False)
        self.fc.weight = self.token_embedding.weight      # 重み共有
        self.apply(self._init_weights)     # 埋め込み部分の初期重み変更

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len,device=x.device)

        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)
        x = tok_emb + pos_emb
        x = self.dropout(x)
        # nn.Transformer.generate_square_subsequent_mask を使ってマスクを生成
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, dtype=torch.bool, device=x.device)

        # 自己回帰型 transformer (transformer decoder)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        x = self.layer_norm(x)
        
        # NTP：次のトークン予測
        logits = self.fc(x)
        return logits

## 指示チューニング用に変数を更新
* 事前学習の重みの読み込み

In [5]:
checkpoint = torch.load(pretrain_filename, map_location=device)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])

# configの追加設定と数値の変更
# update関数を使ってSFT向けに上書き
config.update(
    batch_size    = 128,
    learning_rate = 3e-4,
    num_epochs    = 60,
)

model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [6]:
class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, config):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = config.seq_len

        # 特殊トークンID
        self.system_id    = tokenizer.token_to_id("<system>")
        self.user_id      = tokenizer.token_to_id("<user>")
        self.assistant_id = tokenizer.token_to_id("<assistant>")
        self.eod_id       = tokenizer.token_to_id("<eod>")

    def __len__(self):
        return len(self.data)

    def _enc(self, text):
        return self.tokenizer.encode(text, add_special_tokens=False).ids

    def __getitem__(self, idx):
        item = self.data.iloc[idx]   # データフレームのindex行を取得したいので data.iloc[]を使う
        system      = "あなたはいずれ最強のAIです。次の要求を適切に満たす応答を書きなさい。"
        instruction = item["question"]
        response    = item["answer"]

        # プロンプト部分（損失を計算しない）
        prompt_ids = []
        if system:  # 今回は全部このパターン
            prompt_ids += [self.system_id] + self._enc(system)
        prompt_ids += [self.user_id] + self._enc(instruction)
        prompt_ids += [self.assistant_id]   # ここまで与えて、続きを生成させる

        # 応答部分（損失を計算する）。末尾に<eod>を付けて「停止」を学習させる
        response_ids = self._enc(response) + [self.eod_id]

        input_ids = prompt_ids + response_ids
        # プロンプト部分は-100でマスク、応答部分だけ学習対象
        labels = [-100] * len(prompt_ids) + response_ids

        # seq_lenで切り詰め（長すぎる応答は<eod>が切れる点に注意）
        # seq_len=512なので多分大丈夫
        input_ids = input_ids[: self.max_len]
        labels    = labels[: self.max_len]

        return {"input_ids": input_ids, "labels": labels}

In [7]:
def make_collate_fn(pad_id):
    def collate_fn(batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        input_ids, labels = [], []
        for b in batch:
            n_pad = max_len - len(b["input_ids"])
            input_ids.append(b["input_ids"] + [pad_id] * n_pad)
            labels.append(b["labels"]    + [-100]  * n_pad)  # padは損失対象外
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels":    torch.tensor(labels,    dtype=torch.long),
        }
    return collate_fn

In [8]:
df = pd.read_json(data_filename, lines=True)
dataset = SFTDataset(data=df, tokenizer=tokenizer, config=config)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=config.batch_size,  # メモリ足りない場合は小さくする
    shuffle=True,
    num_workers=0, # CPUの利用コア数みたいなものCPUコアの半分くらい？
    pin_memory=torch.cuda.is_available(), # GPU使う時True
    drop_last=True,
    collate_fn=make_collate_fn(config.pad_token_id),
)

In [9]:
# ByteLevelの時は利用する
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()
decoded = tokenizer.decode(dataset[0]["input_ids"], skip_special_tokens=False)
decoded

'<system>あなたはいずれ最強のAIです。次の要求を適切に満たす応答を書きなさい。<user>新しくオープンする銀座の店舗は、どのような商品ラインナップを展開する予定ですか。<assistant>メンズ、レディスに加え、日本初となるキッズまでフルラインナップを展開する予定です。<eod>'

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=config.ignore_index)   # ignore_index=-100を無視
optimizer = torch.optim.AdamW(model.parameters(),lr=config.learning_rate)

In [ ]:
model.train()
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

for epoch in range(config.num_epochs):
    total_loss = 0.0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels    = batch["labels"].to(device, non_blocking=True)
        optimizer.zero_grad()

        # Flash Attentionを使う
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(input_ids)
            # SFTDatasetクラスではinput_idsとlabelsが１つずれていません　損失計算時に次のトークン予測として１個ずらす作業をします
            shift_logits = logits[:, :-1, :]
            shift_labels = labels[:, 1:]
            loss = criterion(
                shift_logits.reshape(-1, config.vocab_size),
                shift_labels.reshape(-1),
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1)%10 == 0:
        print(f"Epoch {epoch+1}/{config.num_epochs} | Loss: {total_loss/len(dataloader):.4f}")

Epoch 10/60 | Loss: 2.4290
Epoch 20/60 | Loss: 1.4832
Epoch 30/60 | Loss: 0.7496
Epoch 40/60 | Loss: 0.4004
Epoch 50/60 | Loss: 0.2102
Epoch 60/60 | Loss: 0.1155


### 文章生成

In [11]:
# ByteLevelの時は利用する
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()

@torch.inference_mode()
def generate_text(
    model,
    input_ids,
    max_new_tokens=config.seq_len,
    eos_token_id=None,
):
    model.eval()
    generated = input_ids.to(device)
    seq_len = model.config.seq_len

    # greedyで単純に生成
    # eos_token_idで生成終了
    for _ in range(max_new_tokens):
        logits = model(generated[:, -seq_len:])
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return generated

def build_chat_prompt(tokenizer, instruction, system=None):
    ids = []
    ids += [tokenizer.token_to_id("<system>")] + tokenizer.encode(system, add_special_tokens=False).ids
    ids += [tokenizer.token_to_id("<user>")] + tokenizer.encode(instruction, add_special_tokens=False).ids
    ids += [tokenizer.token_to_id("<assistant>")]   # ここから先を生成
    return ids


def response(prompt):
    system_msg = "あなたはいずれ最強のAIです。次の要求を適切に満たす応答を書きなさい。"
    #system_msg = "以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。"

    prompt_ids = build_chat_prompt(tokenizer, prompt, system=system_msg)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)

    output_ids = generate_text(
        model=model, 
        input_ids=input_ids, 
        max_new_tokens=256, 
        eos_token_id=config.eod_token_id,   # <eod>で停止
    )

    # 応答部分だけ取り出す
    response_only = output_ids[0].tolist()[len(prompt_ids):]
    print(tokenizer.decode(response_only, skip_special_tokens=False))

In [14]:
df = pd.read_csv("prompt.csv")

for prompt in df["prompt"]:
    print(f"> {prompt}")
    response(prompt)
    print("\n")

> 親が男の子に就かせたい職業のランキングで1位に挙げられたのはどの職業ですか。
公務員<eod>


> ポール・スミスが発表したコレクションの全体的なコンセプトはどのようなものですか。
英国服の伝統を背景に、斬新なフォルムや素材でドレッシングを楽しむネオ・クラシックなものです。<eod>


> ポール・スミスの「The City」というリストウォッチの価格はいくらですか？"
3万8,850円です。<eod>


> 年収1000万円以上のビジネスパーソンが働いてみたいと思う国は？
シンガポール<eod>


> 転職における市場価値をアップさせるための方法について解説してください。
「仕事が面白くない」・・・若手社会人の悩みは尽きないもの。そんな様々な悩みに辛口4姉妹がお答えします。<eod>


> ノマド男子が持つ「ノマド」という言葉やイメージについて
「ノマド」という言葉だけが一人歩きしているように感じられる点や、SNS上では大きなビジネスをしているように見えがちだが、実際は人間関係が薄い場合がある点です。<eod>


> トヨタ86が世間の注目を集める理由について
「86」という俗称に由来し、「AE86のようにユーザーから愛され、ユーザーが育てる車になって欲しい」という思いが込められているからです。<eod>


> トヨタ86が世間の注目を集める理由はなんですか？
「86」という俗称に由来し、「AE86のようにユーザーから愛され、ユーザーが育てる車になって欲しい」という思いが込められているからです。<eod>


> スマートフォンを所有するユーザーの日常的に利用しているアプリの利用傾向についてまとめてください。
仕事の予定、連絡先、アイデアや企画など、自身の頭の中に全て詰め込めないような情報をスマートフォンに保存するという使い方、いわば「手帳代わり」という用途です。<eod>


> スマートフォンを購入する動機について教えてください。
「PCサイトを閲覧できるから」と「アプリが豊富」という意見でした。<eod>


> マキコさんが経験した、夜景のサプライズの具体的な場所について教えてください。
渋谷の某ホテルの30階くらいで、部屋を予約してくれており、エレベーターに乗って部屋に入ったところ、夜景が目の前にあった。<eod>


> 「原価率」

In [ ]:
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, instruct_model_filename)


In [13]:
checkpoint = torch.load(instruct_model_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

## １ターンの対話をしてみた


In [20]:
for _ in range(5):
    prompt = input(">")
    print(f"> {prompt}")
    response(prompt)
    print("\n")

> こんにちは
こんにちは!<eod>


> トヨタ86について説明して
「86」という俗称に表れから愛され、ユーザーが育てる車になって欲しい。86のようにご存知れた車になって欲しい「86」という思いが愛され、ユーザーが育てる車になって欲しい」という思いが込められているからです。<eod>


> 図解説明とは何でしょうか？
相変わっていくください。<eod>


> スマートフォンの購入理由について教えてください。
「PCサイトを閲覧できるから」と「アプリが豊富」という意見でした。<eod>


> ありがとうございました
こちらこそ、いつもありがとうございます。<eod>


